# Iterative EDT clustering example
This notebook demonstrates the `edt_iterative_clustering()` method on a synthetic CT volume, visualizes the resulting `cluster_map`, and prints computed emphysema indices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from emphy_size_distance_iterative import (
    compute_edt,
    edt_iterative_clustering,
    compute_emphysema_indices,
    noise_reduction,
)

%matplotlib inline

In [ ]:
# Synthetic CT (matches test fixtures)
shape = (50, 128, 128)  # z, y, x
volume = np.full(shape, -1000, dtype=np.float32)  # background air
# Create lung region (central cube)
volume[10:40, 30:98, 30:98] = -500
# Add one spherical emphysema pocket
zc, yc, xc = 20, 64, 64
radius = 8
for z in range(max(0, zc-radius), min(shape[0], zc+radius)):
    for y in range(max(0, yc-radius), min(shape[1], yc+radius)):
        for x in range(max(0, xc-radius), min(shape[2], xc+radius)):
            if np.sqrt((z-zc)**2 + (y-yc)**2 + (x-xc)**2) < radius:
                volume[z,y,x] = -960

# Spacing (z,y,x) in mm
spacing = (2.0, 0.625, 0.625)

# Simple lung and emphysema masks
lung_mask = (volume < -400)
emph_mask = (volume <= -950) & lung_mask

# Clean small noise (use function from module)
cleaned = noise_reduction(emph_mask)
cleaned.sum(), emph_mask.sum()

In [ ]:
# Compute EDT and cluster map
edt = compute_edt(cleaned, spacing)
cluster_map = edt_iterative_clustering(cleaned, edt, spacing)
result = compute_emphysema_indices(cluster_map, lung_mask, cleaned, spacing)
print(result.summary())

In [ ]:
# Visualization: axial, sagittal, coronal with overlay
mid_z = cluster_map.shape[0] // 2
mid_y = cluster_map.shape[1] // 2
mid_x = cluster_map.shape[2] // 2
cmap = ListedColormap(['black', 'cyan', 'yellow', 'magenta', 'red'])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Axial
axes[0].imshow(volume[mid_z], cmap='gray', origin='lower')
axes[0].imshow(cluster_map[mid_z], cmap=cmap, alpha=0.5, origin='lower', vmin=0, vmax=4)
axes[0].set_title(f'Axial Z={mid_z}')
axes[0].axis('off')
# Sagittal (x slice)
axes[1].imshow(np.rot90(volume[:, :, mid_x]), cmap='gray', origin='lower')
axes[1].imshow(np.rot90(cluster_map[:, :, mid_x]), cmap=cmap, alpha=0.5, origin='lower', vmin=0, vmax=4)
axes[1].set_title(f'Sagittal X={mid_x}')
axes[1].axis('off')
# Coronal (y slice)
axes[2].imshow(np.rot90(volume[:, mid_y, :]), cmap='gray', origin='lower')
axes[2].imshow(np.rot90(cluster_map[:, mid_y, :]), cmap=cmap, alpha=0.5, origin='lower', vmin=0, vmax=4)
axes[2].set_title(f'Coronal Y={mid_y}')
axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Save figure to workspace
fig.savefig('iterative_example_cluster_map.png', bbox_inches='tight', dpi=150)
print('Saved: iterative_example_cluster_map.png')

## Next steps
- Run this notebook interactively to inspect different slices and parameters.
- Compare `cluster_map` to outputs from `emphy_size.py` and `emphy_size_distance.py`.